# 02. Bundle synthesis with approximation

This notebook implements the **bundle-of-trajectories** stage:
1. Build a bundle from top-scoring trajectories.
2. Train a quadratic control approximation per time step.
3. Generate trajectories with the approximated feedback law.
4. Evaluate score quality and visualize approximation behavior.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from dynamics import (
    CostConfig,
    build_trajectory_bundle,
    evaluate_closed_loop,
    evaluate_pointwise_rmse,
    load_training_samples,
    select_top_k_trajectories,
    split_trajectory_ids,
    terminal_errors,
    train_quadratic_controllers,
)

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)


In [ ]:
DATA_DIR = ROOT / 'src' / 'data'
samples = load_training_samples(DATA_DIR)
bundle = build_trajectory_bundle(samples)

print('Total step samples:', len(samples))
print('Total unique trajectories:', samples['trajectory_id'].nunique())


In [ ]:
# Build the top-K bundle (best trajectories) for approximation.
TOP_K = 295
selected_ids, selected, score_table = select_top_k_trajectories(samples, bundle, TOP_K)

cfg = CostConfig()
N_STEPS = cfg.num_intervals

print('All trajectories with score:', len(score_table))
print(score_table.head(10))
print('Selected trajectories:', len(selected))
print('Average score in selected bundle:', np.mean([selected[tid]['score'] for tid in selected]))
print('Worst score in selected bundle:', np.max([selected[tid]['score'] for tid in selected]))


In [ ]:
# Split by trajectory ID to avoid leakage across steps.
train_ids, test_ids = split_trajectory_ids(selected_ids, train_ratio=0.8, seed=42)

print('Train trajectories:', len(train_ids))
print('Test trajectories:', len(test_ids))


In [ ]:
FEATURE_DIMS = [6, 5, 4, 3, 2, 1]
RIDGE_LAMBDA = 2e-3

controllers = train_quadratic_controllers(
    selected,
    train_ids,
    N_STEPS,
    feature_dims=FEATURE_DIMS,
    ridge_lambda=RIDGE_LAMBDA,
)

for m in FEATURE_DIMS:
    ctrl = controllers[m]
    print(f'm={m}')
    print('Trained step models:', len(ctrl.models), 'out of', N_STEPS)

controller = controllers[6]


In [ ]:
# Pointwise approximation metrics on held-out trajectories.
rmse_per_control, global_rmse = evaluate_pointwise_rmse(selected, test_ids, controller, N_STEPS)

print('Control RMSE [phi, theta, psi, thrust]:', np.round(rmse_per_control, 5))
print('Global RMSE:', float(global_rmse))


In [ ]:
# Closed-loop synthesis from held-out initial conditions.
MAX_EVAL_TRAJECTORIES = 60
results, res_df = evaluate_closed_loop(
    selected,
    test_ids,
    controller,
    cfg,
    max_cases=MAX_EVAL_TRAJECTORIES,
)

print(res_df[['pred_score', 'true_score']].describe())
print('Mean score gap (pred - true):', float((res_df['pred_score'] - res_df['true_score']).mean()))


## Сравнение эталонных сгенерированных траекторий и траекторий, полученных синтезированным управлением, в плоскости x1-x3 (m=6)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for tid in train_ids[:40]:
    X = selected[int(tid)]['X']
    ax.plot(X[:, 0], X[:, 2], color='#9AA0A6', alpha=0.22, linewidth=1)

for r in results[:20]:
    S = r['states']
    ax.plot(S[:, 0], S[:, 2], color='#E63946', alpha=0.9, linewidth=1.8)

for cyl in cfg.cylinders:
    c = plt.Circle((cyl.x, cyl.z), cyl.radius, color='#F4A261', alpha=0.25)
    ax.add_patch(c)
for wnd in cfg.windows:
    w = plt.Circle((wnd.x, wnd.z), wnd.radius, color='#2A9D8F', alpha=0.35)
    ax.add_patch(w)

ax.scatter([cfg.terminal_state[0]], [cfg.terminal_state[2]], marker='o', s=180, color='black', label='terminal')
ax.set_title('Top-K bundle (gray) and synthesized trajectories (red)')
ax.set_xlabel('x1')
ax.set_ylabel('x3')
ax.axis('equal')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Терминальная ошибка

In [ ]:
term_err = terminal_errors(results, cfg, max_cases=20)

print('Terminal position error for first 20 rollouts:')
print(np.round(term_err, 5))

print('Mean terminal error:', round(float(term_err.mean()), 5))
print('Std terminal error:', round(float(term_err.std()), 5))
print('Max terminal error:', round(float(term_err.max()), 5))
print('Min terminal error:', round(float(term_err.min()), 5))

plt.figure(figsize=(6, 4))
plt.hist(term_err, bins=15)
plt.xlabel('Terminal position error')
plt.ylabel('Count')
plt.title('Distribution of terminal errors')
plt.tight_layout()
plt.show()


### Вывод

Синтезированный контроллер устойчиво воспроизводит целевые траектории и приводит систему в малую окрестность терминального состояния. Средняя ошибка по финальной позиции на 10 тестовых rollout’ах составляет около 0.23 при низком разбросе (std ≈ 0.011).
